In [19]:
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer
from torch.utils.data import Dataset
model_name = "TinyLlama v1.1"

tokenizer = LlamaTokenizer.from_pretrained(model_name)
# ✅ FIX HERE
tokenizer.pad_token = tokenizer.eos_token
model = LlamaForCausalLM.from_pretrained(model_name)



device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
print(f"Using device: {device}")



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14461.38it/s]

Using device: cpu


In [20]:
texts = [
    "User: What is AI?\nAssistant: AI is the simulation of human intelligence.",
    "User: Hello\nAssistant: Hi! How can I help you today?",
    "User: Explain Python\nAssistant: Python is a programming language used for many applications."
]

In [21]:
class ChatDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):  # ✅ add this
        self.data = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        for txt in texts:
            enc = tokenizer(
                txt,
                truncation=True,
                padding="max_length",
                max_length=self.max_length,  # ✅ use self.max_length
                return_tensors="pt"
            )
            self.data.append(enc.input_ids[0])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        return {"input_ids": x, "labels": x.clone()}

In [23]:
from torch.utils.data import DataLoader

dataset = ChatDataset(texts, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

for epoch in range(3):
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"Loss: {loss.item()}")

: 